In [0]:
import sys
sys.path.append('../')
import utils.utilfunc as ut
import Jobcontrol as jc
import datetime

In [0]:
#Job Parameters
rundate = ut.get_rundate()
schema_name = 'warehouse.edw_ld'
table_name = 'dim_date_ld'
table_full_name = f"{schema_name}.{table_name}"
print("Job Triggered for rundate: ",rundate)


In [0]:
# Define or create a datafrane by calling date_data()
_cols = ['date','day','month','year','dayofweek']
_data = ut.date_data(rundate, 2)
df = spark.createDataFrame(_data, schema=_cols)
df.printSchema()

In [0]:
# Add Extra Columns
dfadd = df.selectExpr('*','current_timestamp() as insert_dt',f'{rundate} as rundate')


In [0]:
# Save the Data to Delta Table
if jc.get_max_timestamp(spark,schema_name,table_name) != '1900-01-01 00:00:00.000000':
    dfadd.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_full_name)
else:
    dfadd.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_full_name)


In [0]:
from delta.tables import DeltaTable
dc = DeltaTable.forName(spark, table_full_name)
# Update the Delta Table History
dc.history().limit(1).select("version","operationMetrics.exectuionTimeMs","operationMetrics.numTargetRowsInserted","operationMetrics.numTargetRowsUpdated","operationMetrics.numOutputRows").display()

In [0]:
#Insert into Log Table
jc.insert_log(spark,schema_name,table_name,datetime.datetime.now(),rundate)

In [0]:
%sql
select * from warehouse.edw_ld.dim_date_ld

In [0]:
%sql
select * from warehouse.edw.job_control